In [2]:
"""
==========================================================================
  [Cell 1] 무결성 사전 검증 (Pre-flight Check)
==========================================================================
  
  목적: 24시간 장기 학습 전, 데이터 결함으로 인한 중간 실패를 원천 차단
  
  검증 항목:
    1. 필수 컬럼 존재 여부
    2. 필수 컬럼 결측치(NaN) 검사
    3. 이미지 파일 실존 여부 전수 검사
    4. answer 컬럼 유효값(a/b/c/d) 검증
==========================================================================
"""

import os
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# ──────────────────────────────────────────────
# 경로 설정
# ──────────────────────────────────────────────
BASE_DIR = Path(r"C:\Users\junta\Downloads\2026-ssafy-ai-15-2")
TRAIN_CSV = BASE_DIR / "train_final.csv"

REQUIRED_COLUMNS = ["id", "path", "question", "answer"]

print("=" * 70)
print("  [Pre-flight Check] 데이터 무결성 검증 시작")
print("=" * 70)

# ──────────────────────────────────────────────
# Step 1: CSV 파일 존재 확인
# ──────────────────────────────────────────────
print("\n[1/5] CSV 파일 존재 확인...")
if not TRAIN_CSV.exists():
    raise FileNotFoundError(f"❌ CSV 파일이 존재하지 않습니다: {TRAIN_CSV}")
print(f"  ✅ CSV 파일 확인: {TRAIN_CSV}")

# ──────────────────────────────────────────────
# Step 2: CSV 로드 및 필수 컬럼 존재 확인
# ──────────────────────────────────────────────
print("\n[2/5] CSV 로드 및 필수 컬럼 검증...")
df = pd.read_csv(TRAIN_CSV)
print(f"  총 데이터 수: {len(df):,}건")

missing_cols = [col for col in REQUIRED_COLUMNS if col not in df.columns]
if missing_cols:
    raise ValueError(f"❌ 필수 컬럼 누락: {missing_cols}\n   현재 컬럼: {df.columns.tolist()}")
print(f"  ✅ 필수 컬럼 존재: {REQUIRED_COLUMNS}")

# ──────────────────────────────────────────────
# Step 3: 필수 컬럼 결측치(NaN) 검사
# ──────────────────────────────────────────────
print("\n[3/5] 필수 컬럼 결측치(NaN) 검사...")
nan_report = {}
for col in REQUIRED_COLUMNS:
    nan_count = df[col].isna().sum()
    if nan_count > 0:
        nan_report[col] = nan_count

if nan_report:
    error_msg = "❌ 결측치 발견:\n"
    for col, cnt in nan_report.items():
        error_msg += f"   - {col}: {cnt:,}건 NaN\n"
    raise ValueError(error_msg)
print(f"  ✅ 모든 필수 컬럼에 결측치 없음")

# ──────────────────────────────────────────────
# Step 4: answer 컬럼 유효값 검증
# ──────────────────────────────────────────────
print("\n[4/5] answer 컬럼 유효값 검증...")
valid_answers = {"a", "b", "c", "d"}
df["answer_clean"] = df["answer"].astype(str).str.strip().str.lower()
invalid_answers = df[~df["answer_clean"].isin(valid_answers)]

if len(invalid_answers) > 0:
    invalid_values = invalid_answers["answer_clean"].unique()[:10]
    raise ValueError(
        f"❌ 유효하지 않은 answer 값 발견: {len(invalid_answers):,}건\n"
        f"   허용값: {valid_answers}\n"
        f"   발견된 값 샘플: {invalid_values.tolist()}"
    )
print(f"  ✅ 모든 answer 값이 유효 (a/b/c/d)")
df.drop(columns=["answer_clean"], inplace=True)

# ──────────────────────────────────────────────
# Step 5: 이미지 파일 실존 여부 전수 검사
# ──────────────────────────────────────────────
print("\n[5/5] 이미지 파일 실존 여부 전수 검사...")
missing_images = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="  이미지 검증", unit="file"):
    img_path = Path(row["path"])
    if not img_path.is_absolute():
        img_path = BASE_DIR / img_path
    
    if not img_path.exists():
        missing_images.append({"index": idx, "id": row["id"], "path": str(row["path"])})

if missing_images:
    error_msg = f"❌ 존재하지 않는 이미지 파일: {len(missing_images):,}건\n\n"
    for item in missing_images[:10]:
        error_msg += f"   - Row {item['index']}: {item['path']}\n"
    raise FileNotFoundError(error_msg)

print(f"  ✅ 모든 이미지 파일 존재 확인 완료 ({len(df):,}건)")

# ──────────────────────────────────────────────
# 최종 결과
# ──────────────────────────────────────────────
print("\n" + "=" * 70)
print("  ✅ [Pre-flight Check] 모든 검증 통과!")
print("=" * 70)
print(f"\n  📊 데이터 요약:")
print(f"     - 총 샘플 수: {len(df):,}건")
print(f"\n  📈 Answer 분포:")
for ans, cnt in df["answer"].value_counts().sort_index().items():
    print(f"     - {ans}: {cnt:,}건 ({cnt/len(df)*100:.1f}%)")
print("\n  ➡️  Cell 2 (학습 파이프라인)를 실행하세요.")


  [Pre-flight Check] 데이터 무결성 검증 시작

[1/5] CSV 파일 존재 확인...
  ✅ CSV 파일 확인: C:\Users\junta\Downloads\2026-ssafy-ai-15-2\train_final.csv

[2/5] CSV 로드 및 필수 컬럼 검증...
  총 데이터 수: 60,438건
  ✅ 필수 컬럼 존재: ['id', 'path', 'question', 'answer']

[3/5] 필수 컬럼 결측치(NaN) 검사...
  ✅ 모든 필수 컬럼에 결측치 없음

[4/5] answer 컬럼 유효값 검증...
  ✅ 모든 answer 값이 유효 (a/b/c/d)

[5/5] 이미지 파일 실존 여부 전수 검사...


  이미지 검증: 100%|██████████| 60438/60438 [00:03<00:00, 18124.39file/s]

  ✅ 모든 이미지 파일 존재 확인 완료 (60,438건)

  ✅ [Pre-flight Check] 모든 검증 통과!

  📊 데이터 요약:
     - 총 샘플 수: 60,438건

  📈 Answer 분포:
     - a: 11,724건 (19.4%)
     - b: 25,128건 (41.6%)
     - c: 12,114건 (20.0%)
     - d: 11,472건 (19.0%)

  ➡️  Cell 2 (학습 파이프라인)를 실행하세요.


In [3]:
"""
==========================================================================
  [Cell 1] 무결성 사전 검증 (Pre-flight Check)
==========================================================================
"""

import os
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# ──────────────────────────────────────────────
# 경로 설정
# ──────────────────────────────────────────────
BASE_DIR = Path(r"C:\Users\junta\Downloads\2026-ssafy-ai-15-2")
TRAIN_CSV = BASE_DIR / "train_final.csv"
TEST_CSV = BASE_DIR / "test.csv"
OUTPUT_DIR = BASE_DIR / "vqa_lora_7b"

REQUIRED_COLUMNS = ["id", "path", "question", "answer"]

print("=" * 70)
print("  [Pre-flight Check] 데이터 무결성 검증 시작")
print("=" * 70)

# ──────────────────────────────────────────────
# Step 1: 출력 디렉토리 생성
# ──────────────────────────────────────────────
print("\n[1/6] 출력 디렉토리 생성...")

# 메인 출력 디렉토리
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"  ✅ 메인 디렉토리: {OUTPUT_DIR}")

# 체크포인트 디렉토리
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(f"  ✅ 체크포인트 디렉토리: {CHECKPOINT_DIR}")

# 제출 CSV 저장 디렉토리
SUBMISSION_DIR = OUTPUT_DIR / "submissions"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
print(f"  ✅ 제출 CSV 디렉토리: {SUBMISSION_DIR}")

# 어댑터 저장 디렉토리 (에폭별)
ADAPTER_DIR = OUTPUT_DIR / "adapters"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
print(f"  ✅ 어댑터 디렉토리: {ADAPTER_DIR}")

# ──────────────────────────────────────────────
# Step 2: CSV 파일 존재 확인
# ──────────────────────────────────────────────
print("\n[2/6] CSV 파일 존재 확인...")
if not TRAIN_CSV.exists():
    raise FileNotFoundError(f"❌ Train CSV 없음: {TRAIN_CSV}")
print(f"  ✅ Train CSV: {TRAIN_CSV}")

if not TEST_CSV.exists():
    raise FileNotFoundError(f"❌ Test CSV 없음: {TEST_CSV}")
print(f"  ✅ Test CSV: {TEST_CSV}")

# ──────────────────────────────────────────────
# Step 3: CSV 로드 및 필수 컬럼 검증
# ──────────────────────────────────────────────
print("\n[3/6] CSV 로드 및 필수 컬럼 검증...")
df = pd.read_csv(TRAIN_CSV)
print(f"  Train 데이터 수: {len(df):,}건")

test_df = pd.read_csv(TEST_CSV)
print(f"  Test 데이터 수: {len(test_df):,}건")

missing_cols = [col for col in REQUIRED_COLUMNS if col not in df.columns]
if missing_cols:
    raise ValueError(f"❌ 필수 컬럼 누락: {missing_cols}")
print(f"  ✅ 필수 컬럼 존재: {REQUIRED_COLUMNS}")

# ──────────────────────────────────────────────
# Step 4: 결측치 검사
# ──────────────────────────────────────────────
print("\n[4/6] 필수 컬럼 결측치(NaN) 검사...")
nan_report = {}
for col in REQUIRED_COLUMNS:
    nan_count = df[col].isna().sum()
    if nan_count > 0:
        nan_report[col] = nan_count

if nan_report:
    error_msg = "❌ 결측치 발견:\n"
    for col, cnt in nan_report.items():
        error_msg += f"   - {col}: {cnt:,}건\n"
    raise ValueError(error_msg)
print(f"  ✅ 결측치 없음")

# ──────────────────────────────────────────────
# Step 5: Answer 유효값 검증
# ──────────────────────────────────────────────
print("\n[5/6] answer 컬럼 유효값 검증...")
valid_answers = {"a", "b", "c", "d"}
df["answer_clean"] = df["answer"].astype(str).str.strip().str.lower()
invalid = df[~df["answer_clean"].isin(valid_answers)]

if len(invalid) > 0:
    raise ValueError(f"❌ 유효하지 않은 answer: {len(invalid):,}건")
print(f"  ✅ 모든 answer 유효 (a/b/c/d)")
df.drop(columns=["answer_clean"], inplace=True)

# ──────────────────────────────────────────────
# Step 6: 이미지 파일 전수 검사
# ──────────────────────────────────────────────
print("\n[6/6] 이미지 파일 실존 여부 전수 검사...")
missing_images = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="  Train 이미지", unit="file"):
    img_path = Path(row["path"])
    if not img_path.is_absolute():
        img_path = BASE_DIR / img_path
    if not img_path.exists():
        missing_images.append(str(row["path"]))

if missing_images:
    raise FileNotFoundError(f"❌ 누락 이미지: {len(missing_images):,}건\n샘플: {missing_images[:5]}")
print(f"  ✅ Train 이미지 전체 확인 ({len(df):,}건)")

# Test 이미지도 검사
missing_test = []
for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="  Test 이미지", unit="file"):
    img_path = Path(row["path"])
    if not img_path.is_absolute():
        img_path = BASE_DIR / img_path
    if not img_path.exists():
        missing_test.append(str(row["path"]))

if missing_test:
    raise FileNotFoundError(f"❌ Test 누락 이미지: {len(missing_test):,}건")
print(f"  ✅ Test 이미지 전체 확인 ({len(test_df):,}건)")

# ──────────────────────────────────────────────
# 최종 결과
# ──────────────────────────────────────────────
print("\n" + "=" * 70)
print("  ✅ [Pre-flight Check] 모든 검증 통과!")
print("=" * 70)
print(f"\n  📂 생성된 디렉토리:")
print(f"     - 메인: {OUTPUT_DIR}")
print(f"     - 체크포인트: {CHECKPOINT_DIR}")
print(f"     - 제출 CSV: {SUBMISSION_DIR}")
print(f"     - 어댑터: {ADAPTER_DIR}")
print(f"\n  📊 데이터 요약:")
print(f"     - Train: {len(df):,}건")
print(f"     - Test: {len(test_df):,}건")
print(f"\n  📈 Answer 분포:")
for ans, cnt in df["answer"].value_counts().sort_index().items():
    print(f"     - {ans}: {cnt:,}건 ({cnt/len(df)*100:.1f}%)")
print("\n  ➡️  Cell 2를 실행하세요.")


  [Pre-flight Check] 데이터 무결성 검증 시작

[1/6] 출력 디렉토리 생성...
  ✅ 메인 디렉토리: C:\Users\junta\Downloads\2026-ssafy-ai-15-2\vqa_lora_7b
  ✅ 체크포인트 디렉토리: C:\Users\junta\Downloads\2026-ssafy-ai-15-2\vqa_lora_7b\checkpoints
  ✅ 제출 CSV 디렉토리: C:\Users\junta\Downloads\2026-ssafy-ai-15-2\vqa_lora_7b\submissions
  ✅ 어댑터 디렉토리: C:\Users\junta\Downloads\2026-ssafy-ai-15-2\vqa_lora_7b\adapters

[2/6] CSV 파일 존재 확인...
  ✅ Train CSV: C:\Users\junta\Downloads\2026-ssafy-ai-15-2\train_final.csv
  ✅ Test CSV: C:\Users\junta\Downloads\2026-ssafy-ai-15-2\test.csv

[3/6] CSV 로드 및 필수 컬럼 검증...
  Train 데이터 수: 60,438건
  Test 데이터 수: 5,074건
  ✅ 필수 컬럼 존재: ['id', 'path', 'question', 'answer']

[4/6] 필수 컬럼 결측치(NaN) 검사...
  ✅ 결측치 없음

[5/6] answer 컬럼 유효값 검증...
  ✅ 모든 answer 유효 (a/b/c/d)

[6/6] 이미지 파일 실존 여부 전수 검사...


  Train 이미지: 100%|██████████| 60438/60438 [00:03<00:00, 18711.79file/s]


  ✅ Train 이미지 전체 확인 (60,438건)


  Test 이미지: 100%|██████████| 5074/5074 [00:00<00:00, 17049.05file/s]

  ✅ Test 이미지 전체 확인 (5,074건)

  ✅ [Pre-flight Check] 모든 검증 통과!

  📂 생성된 디렉토리:
     - 메인: C:\Users\junta\Downloads\2026-ssafy-ai-15-2\vqa_lora_7b
     - 체크포인트: C:\Users\junta\Downloads\2026-ssafy-ai-15-2\vqa_lora_7b\checkpoints
     - 제출 CSV: C:\Users\junta\Downloads\2026-ssafy-ai-15-2\vqa_lora_7b\submissions
     - 어댑터: C:\Users\junta\Downloads\2026-ssafy-ai-15-2\vqa_lora_7b\adapters

  📊 데이터 요약:
     - Train: 60,438건
     - Test: 5,074건

  📈 Answer 분포:
     - a: 11,724건 (19.4%)
     - b: 25,128건 (41.6%)
     - c: 12,114건 (20.0%)
     - d: 11,472건 (19.0%)

  ➡️  Cell 2를 실행하세요.


In [ ]:
%pip install Pillow

In [ ]:
# [QLoRA 학습 필수 의존성 패키지 설치]
%pip install peft accelerate bitsandbytes

In [2]:
%pip uninstall -y torch torchvision torchaudio

Note: you may need to restart the kernel to use updated packages.


In [1]:
# 로컬 캐시를 무시하고 원격 저장소에서 CUDA 12.1용 정식 빌드를 강제로 가져옵니다.
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --no-cache-dir

Looking in indexes: https://download.pytorch.org/whl/cu121
Note: you may need to restart the kernel to use updated packages.


In [2]:
# [GPU 상태 검증: 설치 후 CUDA가 날아가지 않았는지 확인]
import torch
print(f"GPU 인식 여부: {torch.cuda.is_available()}")  # True가 출력되어야 함

GPU 인식 여부: True


In [ ]:
"""
[Title] Project 'Hidden-Gem Finder' - Qwen2-VL-2B VQA LoRA Fine-tuning (RTX 5070 12GB Native bf16)
불필요한 양자화(bitsandbytes) 의존성을 제거하고, RTX 5070의 순수 연산 능력을 활용해
bfloat16 환경에서 Qwen2-VL-2B 모델을 메모리 초과(OOM) 없이 안정적으로 학습하는 파이프라인입니다.
"""

import os
import gc
import torch
import torch.nn.functional as F
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from datetime import datetime
from torch.utils.data import Dataset
from transformers import (
    AutoProcessor,
    Qwen2VLForConditionalGeneration,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)
from peft import LoraConfig, get_peft_model

# ──────────────────────────────────────────────
# 환경 설정 및 경고 무시
# ──────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ──────────────────────────────────────────────
# 경로 설정
# ──────────────────────────────────────────────
BASE_DIR = Path(r"C:\Users\junta\Downloads\2026-ssafy-ai-15-2")
TRAIN_CSV = BASE_DIR / "train_final.csv"
TEST_CSV = BASE_DIR / "test.csv"
MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

OUTPUT_DIR = BASE_DIR / "vqa_final"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
SUBMISSION_DIR = OUTPUT_DIR / "submissions"
ADAPTER_DIR = OUTPUT_DIR / "adapters"

for d in [OUTPUT_DIR, CHECKPOINT_DIR, SUBMISSION_DIR, ADAPTER_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ──────────────────────────────────────────────
# 12GB VRAM 대응 초보수적 하이퍼파라미터
# ──────────────────────────────────────────────
NUM_EPOCHS = 3
BATCH_SIZE = 1
GRAD_ACCUM = 16
LEARNING_RATE = 1e-4
SEED = 42
LABEL_SMOOTHING = 0.1
LORA_R = 8
LORA_ALPHA = 16
MIN_PIXELS = 64 * 28 * 28
MAX_PIXELS = 256 * 28 * 28

# ──────────────────────────────────────────────
# 유틸리티 함수
# ──────────────────────────────────────────────
def cleanup():
    """GPU 메모리 누수를 방지하기 위한 가비지 컬렉션 및 캐시 초기화"""
    gc.collect()
    torch.cuda.empty_cache()

def build_prompt(row):
    """VQA 포맷에 맞춘 프롬프트 생성"""
    return (
        f"Question: {row['question']}\n"
        f"a) {row['a']}\nb) {row['b']}\nc) {row['c']}\nd) {row['d']}\n"
        f"Answer:"
    )

# ──────────────────────────────────────────────
# 데이터셋 클래스
# ──────────────────────────────────────────────
class TrainDS(Dataset):
    def __init__(self, df, base):
        self.df = df.reset_index(drop=True)
        self.base = base

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        p = Path(r["path"])
        if not p.is_absolute():
            p = self.base / p
        
        return {
            "messages": [
                {"role": "user", "content": [
                    {"type": "image", "image": str(p)},
                    {"type": "text", "text": build_prompt(r)},
                ]},
                {"role": "assistant", "content": [
                    {"type": "text", "text": str(r["answer"]).strip().lower()},
                ]},
            ],
            "image_path": str(p),
            "answer": str(r["answer"]).strip().lower(),
        }

class TestDS(Dataset):
    def __init__(self, df, base):
        self.df = df.reset_index(drop=True)
        self.base = base

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        p = Path(r["path"])
        if not p.is_absolute():
            p = self.base / p
        
        return {
            "id": r["id"],
            "messages": [
                {"role": "user", "content": [
                    {"type": "image", "image": str(p)},
                    {"type": "text", "text": build_prompt(r)},
                ]},
            ],
            "image_path": str(p),
        }

# ──────────────────────────────────────────────
# Collator (배치 전처리)
# ──────────────────────────────────────────────
class Collator:
    def __init__(self, proc):
        self.proc = proc
        self.ans_ids = {}
        for c in ["a", "b", "c", "d"]:
            self.ans_ids[c] = proc.tokenizer.encode(c, add_special_tokens=False)[-1]

    def __call__(self, batch):
        if isinstance(batch, dict):
            batch = [batch]

        fulls, prompts, imgs, answers = [], [], [], []

        for b in batch:
            full = self.proc.apply_chat_template(b["messages"], tokenize=False, add_generation_prompt=False)
            fulls.append(full)

            user = [m for m in b["messages"] if m["role"] != "assistant"]
            prompt = self.proc.apply_chat_template(user, tokenize=False, add_generation_prompt=True)
            prompts.append(prompt)

            imgs.append(Image.open(b["image_path"]).convert("RGB"))
            answers.append(b["answer"])

        inputs = self.proc(text=fulls, images=imgs, padding=True, return_tensors="pt")
        prompt_lens = [len(self.proc(text=[p], images=[img], return_tensors=None)["input_ids"][0]) 
                       for p, img in zip(prompts, imgs)]

        labels = inputs["input_ids"].clone()
        pad = self.proc.tokenizer.pad_token_id
        if pad:
            labels[labels == pad] = -100

        info = []
        for i, plen in enumerate(prompt_lens):
            if pad:
                mask = inputs["input_ids"][i] != pad
                start = mask.nonzero(as_tuple=True)[0][0].item() if mask.any() else 0
            else:
                start = 0
            labels[i, start:start+plen] = -100
            info.append({"i": i, "pos": start+plen, "ans": answers[i]})

        inputs["labels"] = labels
        inputs["info"] = info
        inputs["aids"] = self.ans_ids
        return inputs

# ──────────────────────────────────────────────
# Custom Trainer (Loss 커스텀)
# ──────────────────────────────────────────────
class MyTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        info = inputs.pop("info", [])
        aids = inputs.pop("aids", {})
        
        out = model(**inputs)
        logits, labels = out.logits, inputs["labels"]
        
        sl = logits[..., :-1, :].contiguous()
        slb = labels[..., 1:].contiguous()
        
        loss = F.cross_entropy(sl.view(-1, sl.size(-1)), slb.view(-1), ignore_index=-100)
        
        if info:
            _, seq, vocab = sl.shape
            smooth = torch.tensor(0.0, device=logits.device, dtype=logits.dtype)
            for x in info:
                pos = x["pos"] - 1
                if 0 <= pos < seq:
                    tgt = torch.zeros(vocab, device=logits.device, dtype=logits.dtype)
                    for c, tid in aids.items():
                        tgt[tid] = 0.9 if c == x["ans"] else 0.033
                    smooth += F.kl_div(F.log_softmax(sl[x["i"], pos], -1), tgt, reduction='sum')
            loss = 0.5 * loss + 0.5 * smooth / max(len(info), 1)
        
        return (loss, out) if return_outputs else loss

# ──────────────────────────────────────────────
# 추론 및 콜백
# ──────────────────────────────────────────────
def infer(model, proc, df):
    model.eval()
    aids = {c: proc.tokenizer.encode(c, add_special_tokens=False)[-1] for c in "abcd"}
    tids = [aids[c] for c in "abcd"]
    
    ds = TestDS(df, BASE_DIR)
    res = []
    
    for i in tqdm(range(len(ds)), desc="추론"):
        item = ds[i]
        txt = proc.apply_chat_template(item["messages"], tokenize=False, add_generation_prompt=True)
        img = Image.open(item["image_path"]).convert("RGB")
        
        inp = proc(text=[txt], images=[img], return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            out = model(**inp)
        
        pos = inp["attention_mask"][0].sum().item() - 1
        scores = [out.logits[0, pos, t].item() for t in tids]
        ans = "abcd"[scores.index(max(scores))]
        
        res.append({"id": item["id"], "answer": ans})
        
        del inp, out
        if i % 100 == 0:
            cleanup()
    
    return pd.DataFrame(res)

class CB(TrainerCallback):
    def __init__(self, proc, tdf):
        self.proc = proc
        self.tdf = tdf
    
    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        ep = int(state.epoch)
        cleanup()
        
        p = ADAPTER_DIR / f"ep{ep}"
        p.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(str(p))
        
        df = infer(model, self.proc, self.tdf)
        
        csv = SUBMISSION_DIR / f"sub_ep{ep}_{datetime.now().strftime('%H%M')}.csv"
        df.to_csv(csv, index=False)
        
        print(f"\n[Epoch {ep}] 저장: {csv.name}")
        print(df["answer"].value_counts().sort_index().to_string())
        
        cleanup()
        return control

# ──────────────────────────────────────────────
# 메인 실행 루프
# ──────────────────────────────────────────────
def main():
    cleanup()
    torch.manual_seed(SEED)
    
    print("=" * 50)
    print("학습 시작")
    print("=" * 50)
    
    df = pd.read_csv(TRAIN_CSV)
    tdf = pd.read_csv(TEST_CSV)
    tr = df.sample(frac=0.95, random_state=SEED)
    ev = df.drop(tr.index)
    
    steps = (len(tr) // (BATCH_SIZE * GRAD_ACCUM)) * NUM_EPOCHS
    print(f"Train: {len(tr)} | Test: {len(tdf)} | Steps: {steps}")
    
    proc = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
    
    cleanup()
    # 핵심 변경점: 4-bit 양자화를 제외하고 bfloat16을 직접 할당하여 모델 로드
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="sdpa",
        low_cpu_mem_usage=True,
    )
    
    # 12GB VRAM을 고려한 메모리 절약 기법 강제 활성화
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    ))
    
    print(f"Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    
    trainer = MyTrainer(
        model=model,
        args=TrainingArguments(
            output_dir=str(CHECKPOINT_DIR),
            num_train_epochs=NUM_EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM,
            optim="adamw_torch",
            learning_rate=LEARNING_RATE,
            lr_scheduler_type="cosine",
            warmup_steps=int(steps * 0.03),
            bf16=True, # bfloat16 활성화
            logging_steps=200,
            save_strategy="epoch",
            eval_strategy="epoch",
            save_total_limit=1,
            dataloader_num_workers=0,
            dataloader_pin_memory=False,
            remove_unused_columns=False,
            report_to="none",
            gradient_checkpointing=True,
        ),
        train_dataset=TrainDS(tr, BASE_DIR),
        eval_dataset=TrainDS(ev, BASE_DIR),
        data_collator=Collator(proc),
        callbacks=[CB(proc, tdf)],
    )
    
    print("\n학습 시작 (약 8~10시간 예상)")
    print("-" * 50)
    
    try:
        trainer.train()
    except Exception as e:
        print(f"\n에러 발생: {e}")
        print("현재까지 저장된 체크포인트를 확인하십시오.")
        trainer.model.save_pretrained(str(ADAPTER_DIR / "emergency"))
    
    trainer.model.save_pretrained(str(ADAPTER_DIR / "final"))
    proc.save_pretrained(str(ADAPTER_DIR / "final"))
    
    print("\n" + "=" * 50)
    print("완료!")
    print("=" * 50)

if __name__ == "__main__":
    main()

학습 시작
Train: 57416 | Test: 5074 | Steps: 10764


Loading weights: 100%|██████████| 729/729 [00:01<00:00, 539.47it/s]


RuntimeError: CUDA error: no kernel image is available for execution on the device
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
